In [5]:
import os
import pandas as pd

folder_path = r"raw data\dynamic_events_pl_24"

# List all parquet files
files = [f for f in os.listdir(folder_path) if f.lower().endswith(".parquet")]
print("Parquet files found:", files)

for filename in files:
    file_path = os.path.join(folder_path, filename)
    df_name = os.path.splitext(filename)[0]  # remove .parquet extension
    try:
        globals()[df_name] = pd.read_parquet(file_path)
        print(f"DataFrame '{df_name}' created with shape {globals()[df_name].shape}")
    except Exception as e:
        print(f"Error reading {filename}: {e}")

Parquet files found: ['matches.parquet', 'players.parquet', 'teams.parquet']
DataFrame 'matches' created with shape (380, 8)
DataFrame 'players' created with shape (686, 7)
DataFrame 'teams' created with shape (20, 4)


In [9]:
# Expand 'home_team' and 'away_team' dictionaries
home_df = pd.json_normalize(matches['home_team']).add_prefix('home_')
away_df = pd.json_normalize(matches['away_team']).add_prefix('away_')

# Drop the original dict columns and concatenate the new ones
matches_expanded = pd.concat([matches.drop(['home_team', 'away_team'], axis=1), home_df, away_df], axis=1)

matches_expanded.head()

,id,date_time,status,competition_id,season_id,competition_edition_id,home_id,home_short_name,away_id,away_short_name
0,2018571,2025-05-25T15:00:00Z,closed,1,95,895,63,Bournemouth,62,Leicester
1,2018572,2025-05-25T15:00:00Z,closed,1,95,895,48,Fulham,40,Manchester City
2,2018573,2025-05-25T15:00:00Z,closed,1,95,895,752,Ipswich Town,37,West Ham
3,2018574,2025-05-25T15:00:00Z,closed,1,95,895,2,Liverpool,60,Crystal Palace
4,2018575,2025-05-25T15:00:00Z,closed,1,95,895,31,Manchester U,39,Aston Villa


In [7]:
players.head()

,id,first_name,last_name,short_name,birthday,trackable_object,gender
0,18589,Max,Aarons,M. Aarons,2000-01-04,18786,male
1,967750,Zach,Abbott,Z. Abbott,2006-05-13,969313,male
2,514681,Joshua Kofi,Acheampong,J. Acheampong,2006-05-05,516238,male
3,24629,Tyler,Adams,T. Adams,1999-02-14,25664,male
4,11589,Oluwatosin,Adarabioyo,T. Adarabioyo,1997-09-24,11670,male


In [10]:
# Expand the 'stadium' dictionary
stadium_df = pd.json_normalize(teams['stadium']).add_prefix('stadium_')

# Drop the original 'stadium' column and concatenate the new ones
teams_expanded = pd.concat([teams.drop('stadium', axis=1), stadium_df], axis=1)

teams_expanded.head()

,id,name,coach,stadium_capacity,stadium_city,stadium_id,stadium_name
0,39,Aston Villa,None,42682.0,Birmingham,488,Villa Park
1,754,Brentford FC,None,NaN,London,1970,Brentford Community Stadium
2,37,West Ham United,None,60000.0,London,52,Stade Olympique de Londres
3,31,Manchester United,None,75635.0,Manchester,63,Old Trafford
4,41,Everton,None,39571.0,Liverpool,89,Goodison Park


In [11]:
import os
import pandas as pd

# Base folder
base_folder = r"raw data"

# Folders to read
folders = ["dynamic", "meta"]

for folder in folders:
    folder_path = os.path.join(base_folder, folder)
    
    # List all Parquet files in this folder (recursive if needed)
    for root, dirs, files in os.walk(folder_path):  # os.walk allows subfolders too
        for filename in files:
            if filename.lower().endswith(".parquet"):
                file_path = os.path.join(root, filename)
                df_name = os.path.splitext(filename)[0]  # Remove .parquet
                
                # Replace any spaces or illegal characters with underscore for variable names
                df_name = df_name.replace(" ", "_").replace("-", "_")
                
                try:
                    globals()[df_name] = pd.read_parquet(file_path)
                    print(f"Loaded: {df_name} from {folder}")
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

In [20]:
for file in all_files:
    try:
        with open(file, "r", encoding="utf-8") as f:  # specify UTF-8
            data = json.load(f)

        match_flat = pd.json_normalize(data, sep="_", max_level=2)
        match_list.append(match_flat)

    except Exception as e:
        print(f"Error reading {file.name}: {e}")


# Combine all JSONs into one DataFrame, one row per JSON
if match_list:  # check that at least one JSON was loaded
    matches_df = pd.concat(match_list, ignore_index=True)
    print("All matches combined. Shape:", matches_df.shape)
    display(matches_df.head())
else:
    print("No JSON files were successfully loaded.")

All matches combined. Shape: (1026, 69)


,id,home_team_score,away_team_score,date_time,match_periods,referees,players,status,home_team_side,pitch_length,...,competition_edition_season_end_year,competition_edition_season_name,competition_edition_name,competition_round_id,competition_round_name,competition_round_round_number,competition_round_potential_overtime,ball_trackable_object,home_team_coach,away_team_coach
0,1650385,1,0,2024-08-16T19:00:00Z,"[{'period': 1, 'name': 'period_1', 'start_fram...",[],"[{'player_role': {'id': 19, 'position_group': ...",closed,"[left_to_right, right_to_left]",105,...,2025,2024/2025,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN
1,1650961,0,2,2024-08-17T11:30:00Z,"[{'period': 1, 'name': 'period_1', 'start_fram...",[],"[{'player_role': {'id': 20, 'position_group': ...",closed,"[left_to_right, right_to_left]",106,...,2025,2024/2025,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN
2,1651701,2,0,2024-08-17T14:00:00Z,"[{'period': 1, 'name': 'period_1', 'start_fram...",[],"[{'player_role': {'id': 3, 'position_group': '...",closed,"[left_to_right, right_to_left]",105,...,2025,2024/2025,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN
3,1651702,0,3,2024-08-17T14:00:00Z,"[{'period': 1, 'name': 'period_1', 'start_fram...",[],"[{'player_role': {'id': 11, 'position_group': ...",closed,"[right_to_left, left_to_right]",101,...,2025,2024/2025,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN
4,1651704,1,1,2024-08-17T14:00:00Z,"[{'period': 1, 'name': 'period_1', 'start_fram...",[],"[{'player_role': {'id': 13, 'position_group': ...",closed,"[left_to_right, right_to_left]",105,...,2025,2024/2025,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN


In [21]:
# Example for players column
players_df = pd.json_normalize(matches_df['players'].explode(), sep="_")
players_df.head()

,start_time,end_time,number,yellow_card,red_card,injured,goal,own_goal,team_player_id,team_id,...,player_role_name,player_role_acronym,playing_time_total_minutes_tip,playing_time_total_minutes_otip,playing_time_total_start_frame,playing_time_total_end_frame,playing_time_total_minutes_played,playing_time_total_minutes_played_regular_time,playing_time_by_period,playing_time_total
0,00:00:00,None,33,0,0,False,0,0,33982,48,...,Left Back,LB,25.15,27.94,10.0,57549.0,95.88,95.88,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN
1,00:00:00,01:00:11,7,1,0,False,0,0,1062969,31,...,Attacking Midfield,AM,20.42,14.54,10.0,36849.0,61.38,61.38,"[{'name': 'period_1', 'minutes_tip': 17.09, 'm...",NaN
2,00:00:00,01:03:02,32,0,0,False,0,0,1339102,48,...,Center Forward,CF,15.31,20.45,10.0,38559.0,64.23,64.23,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN
3,00:00:00,01:30:53,18,1,0,False,0,0,239802,48,...,Right Midfield,RM,24.20,26.63,10.0,55269.0,92.08,92.08,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN
4,00:00:00,None,2,0,0,False,0,0,33697,48,...,Right Back,RB,25.15,27.94,10.0,57549.0,95.88,95.88,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN
